In [1]:
import pandas as pd
import numpy as np

In [2]:
df_antigo = pd.read_csv('../../dados/raw/clientes.csv', sep=';')
df_novo = pd.read_csv('../../dados/raw/clientes_novos.csv', sep=';')

In [3]:
df = pd.concat(
    [df_antigo, df_novo],
    ignore_index=True
)
df

,id_cliente,nome,cidade_origem,estado_origem,faixa_etaria,tipo_cliente
0,1,Otavio Reis,São Paulo,SP,18-25,PJ
1,2,Larissa Sousa,Curitiba,SC,18-25,pessoa fisica
2,3,Tatiana Coelho,Rio de Janeiro,CE,65+,Corporativo
3,4,Wilson Duarte,Brasília,AM,26-35,corporativo
4,5,Gabriela Lima,Rio de Janeiro,SP,46-55,Pessoa Física
...,...,...,...,...,...,...
316,316,Priscila Guimarães,Lisboa,EX,NaN,PF
317,317,Rodrigo Vasques,Rio de Janeiro,RJ,36-45,PF
318,318,Sabrina Leitão,Brasília,DF,46-55,PF
319,319,Thiago Bicalho,Belo Horizonte,MG,46-55,Pessoa Jurídica


In [4]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 321 entries, 0 to 320
Data columns (total 6 columns):
 #   Column         Non-Null Count  Dtype
---  ------         --------------  -----
 0   id_cliente     321 non-null    int64
 1   nome           321 non-null    str  
 2   cidade_origem  321 non-null    str  
 3   estado_origem  321 non-null    str  
 4   faixa_etaria   290 non-null    str  
 5   tipo_cliente   321 non-null    str  
dtypes: int64(1), str(5)
memory usage: 15.2 KB


### Tratamento cidade origem

In [5]:
df['cidade_origem'] = df['cidade_origem'].str.strip().str.title().str.replace('De', 'de', regex=False) # Caso especial apenas para que fique Rio de Janeiro invés de Rio De Janeiro
df.head(10)

,id_cliente,nome,cidade_origem,estado_origem,faixa_etaria,tipo_cliente
0,1,Otavio Reis,São Paulo,SP,18-25,PJ
1,2,Larissa Sousa,Curitiba,SC,18-25,pessoa fisica
2,3,Tatiana Coelho,Rio de Janeiro,CE,65+,Corporativo
3,4,Wilson Duarte,Brasília,AM,26-35,corporativo
4,5,Gabriela Lima,Rio de Janeiro,SP,46-55,Pessoa Física
5,6,Ulisses Brito,Brasília,RS,56-65,Pessoa Física
6,7,Kleber Pires,São Paulo,ES,65+,pessoa fisica
7,8,Olivia Freitas,Brasília,PA,18-25,Corporativo
8,9,Wesley Araujo,Belo Horizonte,ES,NaN,Corporativo
9,10,Ivan Azevedo,Curitiba,PE,26-35,pf


In [6]:
df['cidade_origem'].value_counts().reset_index()

,cidade_origem,count
0,Fortaleza,46
1,São Paulo,42
2,Recife,41
3,Curitiba,40
4,Brasília,36
5,Belo Horizonte,36
6,Salvador,36
7,Rio de Janeiro,34
8,Lisboa,2
9,Buenos Aires,2


In [7]:
map_estado = {
    'Fortaleza': 'CE',
    'São Paulo': 'SP',
    'Recife': 'PE',
    'Curitiba': 'PR',
    'Brasília': 'DF',
    'Belo Horizonte': 'MG',
    'Salvador': 'BA',
    'Rio de Janeiro': 'RJ',
    'Porto Alegre': 'RS',
    'Niterói': 'RJ'
}

In [8]:
for cidade, estado  in map_estado.items():
    df.loc[df['cidade_origem'] == cidade, 'estado_origem'] = estado

In [9]:
df[['cidade_origem', 'estado_origem']]

,cidade_origem,estado_origem
0,São Paulo,SP
1,Curitiba,PR
2,Rio de Janeiro,RJ
3,Brasília,DF
4,Rio de Janeiro,RJ
...,...,...
316,Lisboa,EX
317,Rio de Janeiro,RJ
318,Brasília,DF
319,Belo Horizonte,MG


### Tratamento tipo de cliente

In [10]:
df['tipo_cliente'] = df['tipo_cliente'].str.strip().str.title()
map_cliente = {
    'Pf': 'Pessoa Física',
    'Pessoa Fisica': 'Pessoa Física',
    'Pj': 'Corporativo',
    'Pessoa Jurídica': 'Corporativo'
}
df['tipo_cliente'] = df['tipo_cliente'].map(map_cliente).fillna(df['tipo_cliente'])

In [11]:
df['tipo_cliente'].value_counts().reset_index()

,tipo_cliente,count
0,Corporativo,171
1,Pessoa Física,150


### Tratamento de nome

In [12]:
df['nome'] = df['nome'].str.strip().str.title()

In [13]:
mascara = df['nome'].str.startswith(' ') | df['nome'].str.endswith(' ')
df.loc[mascara, 'nome'].tolist()

[]

### Verificando faixa etária:

In [14]:
df['faixa_etaria'].value_counts().reset_index().sort_values(by='faixa_etaria')

,faixa_etaria,count
5,18-25,43
4,26-35,44
3,36-45,46
1,46-55,50
2,56-65,47
0,65+,60


### Correções: 

#### Conflito ID:

In [15]:
df.loc[(df['nome'] == 'Carlos Neto') & (df['id_cliente'] == 47), 'id_cliente'] = 401

#### Faixa etária: 

In [16]:
df['faixa_etaria'] = df['faixa_etaria'].fillna('Não Informado')

### Nulos e POSSÍVEIS duplicatas

In [17]:
df.isnull().sum().reset_index()

,index,0
0,id_cliente,0
1,nome,0
2,cidade_origem,0
3,estado_origem,0
4,faixa_etaria,0
5,tipo_cliente,0


In [18]:
df[df.isnull() 
.any(axis=1)]

,id_cliente,nome,cidade_origem,estado_origem,faixa_etaria,tipo_cliente


In [19]:
df[df.duplicated(subset=['nome', 'cidade_origem', 'estado_origem', 'faixa_etaria', 'tipo_cliente'], keep=False)].sort_values(by='nome')

,id_cliente,nome,cidade_origem,estado_origem,faixa_etaria,tipo_cliente
30,31,Abel Correia,Rio de Janeiro,RJ,46-55,Pessoa Física
57,58,Abel Correia,Rio de Janeiro,RJ,46-55,Pessoa Física
37,38,Daniela Ramos,São Paulo,SP,18-25,Pessoa Física
153,154,Daniela Ramos,São Paulo,SP,18-25,Pessoa Física
136,137,Helena Borges,Salvador,BA,36-45,Pessoa Física
255,256,Helena Borges,Salvador,BA,36-45,Pessoa Física
14,15,Henrique Rocha,Recife,PE,Não Informado,Pessoa Física
112,113,Henrique Rocha,Recife,PE,Não Informado,Pessoa Física
24,25,Olivia Freitas,Curitiba,PR,18-25,Corporativo
162,163,Olivia Freitas,Curitiba,PR,18-25,Corporativo


In [20]:
df = df.drop_duplicates(subset=['nome', 'cidade_origem', 'estado_origem', 'faixa_etaria', 'tipo_cliente'], keep='first')

In [21]:
df[df.duplicated(subset=['nome', 'cidade_origem', 'estado_origem', 'faixa_etaria', 'tipo_cliente'], keep=False)].sort_values(by='nome')

,id_cliente,nome,cidade_origem,estado_origem,faixa_etaria,tipo_cliente


In [22]:
df.to_csv('../../dados/clean/clientes.csv', sep=';', encoding='UTF-8', decimal='.')